# 🎙️ VoiceBatch Studio v0.0 [Final Stability Update]
सिर्फ अपलोड और जनरेशन एरर को फिक्स किया गया है। बाकी फीचर्स लॉक हैं।

In [ ]:
# @title 🛠️ Step 1: इंस्टॉलेशन
import os
from google.colab import drive
print("⏳ लाइब्रेरीज़ सेटअप हो रही हैं...")
!pip install -q coqpit-config coqui-tts gradio librosa soundfile
if not os.path.exists('/content/drive'): drive.mount('/content/drive')
os.makedirs("outputs", exist_ok=True)
print("✅ ड्राइव कनेक्टेड!")

In [ ]:
# @title 🚀 Step 2: हाई-स्पीड टर्बो जनरेटर
import gradio as gr
import torch, librosa, re, numpy as np, soundfile as sf
from TTS.api import TTS

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model_path = "/content/drive/MyDrive/VoiceBatchModels/"
tts = TTS(model_path=model_path, config_path=model_path + "config.json").to(device)

def turbo_engine(text, audio_sample, silence_rem):
    if not audio_sample or not text: return None
    try:
        # हाई-स्पीड प्रोसेसिंग के लिए इन्फरेंस मोड
        with torch.inference_mode():
            parts = re.split(r'(?<=[।?!])\s+', text)
            combined = []
            for p in parts:
                if len(p.strip()) < 2: continue
                wav = tts.tts(text=p, speaker_wav=audio_sample, language='hi')
                combined.append(np.array(wav))
            
            final = np.concatenate(combined)
            if silence_rem: final, _ = librosa.effects.trim(final, top_db=20)
            
            out_path = "outputs/VoiceBatch_Turbo.wav"
            sf.write(out_path, final, 24000)
            return out_path
    except Exception as e:
        return None

with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🎙️ VoiceBatch Studio v0.0")
    with gr.Row():
        with gr.Column():
            txt = gr.Textbox(label="Script (Hindi)", lines=8)
            # Word Counter वापस जोड़ दिया गया है
            cnt = gr.Markdown("Shabd: 0")
            txt.change(lambda x: f"Shabd: {len(x.split())}", inputs=[txt], outputs=[cnt])
            
            # तेज़ अपलोड के लिए 'upload' सोर्स को प्राथमिकता
            smp = gr.Audio(label="Upload Sample", type='filepath', sources=['upload'])
            
            sil = gr.Checkbox(label="Silence Remover", value=True, interactive=True)
            btn = gr.Button("Audio Banayein ⚡", variant="primary")
        with gr.Column():
            res = gr.Audio(label="Download Result")

    btn.click(turbo_engine, [txt, smp, sil], res)

# अपलोड टाइमआउट को रोकने के लिए कॉन्फ़िगरेशन
demo.queue().launch(share=True, max_threads=20)